# Kalman Macro Event V4 — Free FRED/ALFRED, 2017→Now\n\nUses free FRED/ALFRED initial-release vintages plus free daily rates context. No paid Trading Economics dependency.\n\n**Research only:** Toss execution OFF / Neon write OFF / production promotion OFF.\n

In [ ]:
# KALMAN Macro Event V4 — one-cell Colab runner
# Research only: Toss OFF / Neon write OFF / production promotion OFF

from google.colab import drive
import json, os, shutil, subprocess, sys
from datetime import datetime, timezone
from pathlib import Path

PINNED_SHA = "215a7d19f2b465ad105b5e0df6f190b1930f47bf"
SOURCE_BRANCH = "feature/macro-event-v1-20260916"
REPO = Path("/content/Codex_macro_v4")
DRIVE_ROOT = Path("/content/drive/MyDrive")
MACRO_INPUT = DRIVE_ROOT / "Market_Macro/v1/raw/us_macro_events_normalized.parquet"
RATES_CONTEXT = DRIVE_ROOT / "Market_Macro/v1/rates/fred_rates_context.parquet"
V4_TAG = "20260916_macro_event_v4_001"

def run(cmd, *, cwd=None, env=None):
    cmd = [str(x) for x in cmd]
    print("\n$", " ".join(cmd))
    subprocess.run(cmd, check=True, cwd=cwd, env=env)

drive.mount("/content/drive", force_remount=False)

if REPO.exists():
    shutil.rmtree(REPO)

run([
    "git", "clone", "--branch", SOURCE_BRANCH,
    "https://github.com/kimtk94/Codex.git", REPO
])
run(["git", "-C", REPO, "checkout", "--detach", PINNED_SHA])
checked = subprocess.check_output(
    ["git", "-C", REPO, "rev-parse", "HEAD"], text=True
).strip()
assert checked == PINNED_SHA, (checked, PINNED_SHA)

APP = REPO / "kalman-toss-gateway"

run([
    sys.executable, "-m", "pip", "install", "-q",
    "pandas", "numpy", "pyarrow", "scikit-learn", "requests"
])

# Reuse existing free FRED/ALFRED outputs when present.
# Otherwise collect 2017->today using the user's free FRED API key.
if not MACRO_INPUT.exists() or not RATES_CONTEXT.exists():
    try:
        from google.colab import userdata
        fred_key = (userdata.get("FRED_API_KEY") or "").strip()
    except Exception:
        fred_key = ""

    if not fred_key:
        from getpass import getpass
        print(
            "FRED_API_KEY was not found in Colab Secrets. "
            "Enter it once below; input is hidden and is not saved to Drive/Git."
        )
        fred_key = getpass("FRED API key: ").strip()

    if not fred_key:
        raise RuntimeError(
            "FRED_API_KEY is required because free FRED/ALFRED backfill files are absent."
        )

    MACRO_INPUT.parent.mkdir(parents=True, exist_ok=True)
    RATES_CONTEXT.parent.mkdir(parents=True, exist_ok=True)
    env = os.environ.copy()
    env["FRED_API_KEY"] = fred_key
    run([
        sys.executable, "-m",
        "research.macro_event.collect_fred_alfred",
        "--start", "2017-01-01",
        "--end", datetime.now(timezone.utc).date().isoformat(),
        "--events-output", MACRO_INPUT,
        "--rates-output", RATES_CONTEXT,
    ], cwd=APP, env=env)

run([
    sys.executable,
    APP / "scripts/colab_macro_v4.py",
    "--drive-root", DRIVE_ROOT,
    "--macro-input", "Market_Macro/v1/raw/us_macro_events_normalized.parquet",
    "--rates-context", "Market_Macro/v1/rates/fred_rates_context.parquet",
    "--v3-candidate-tag", "20260913_return_regime_v3_001",
    "--v4-candidate-tag", V4_TAG,
    "--code-sha", PINNED_SHA,
], cwd=APP)

comparison = (
    DRIVE_ROOT / "Market_Model_V2"
    / "historical_quant_2017_v4_macro_candidate"
    / V4_TAG / "v3_vs_v4_common_window.json"
)
print("\n" + "=" * 88)
print("MACRO V4 COMPLETE")
print("=" * 88)
print("PINNED_SHA:", PINNED_SHA)
print("MACRO_INPUT:", MACRO_INPUT)
print("RATES      :", RATES_CONTEXT)
print("COMPARISON :", comparison)
print("SAFETY     : research-only / Toss OFF / Neon write OFF")
print("\nV3 vs V4:")
print(comparison.read_text(encoding="utf-8"))
